# Week 4: Supervised Learning & Classification
## Notebook 12: Multinomial Logistic (Softmax) Regression from Scratch

### 1. Conceptual Intuition & Theoretical Background
While standard Binary Logistic Regression predicts probabilities for two mutually exclusive classes, **Multinomial Logistic Regression (Softmax Regression)** models $K > 2$ classes simultaneously.

For class $k \in \{1, \dots, K\}$, logits are mapped via the **Softmax Function**:
$$P(Y = k \mid X) = \frac{\exp(X W_k)}{\sum_{j=1}^{K} \exp(X W_j)}$$

Trained by minimizing the **Categorical Cross-Entropy Loss**:
$$\mathcal{L}(W) = -\frac{1}{N} \sum_{i=1}^{N} \sum_{k=1}^{K} y_{ik} \ln \hat{p}_{ik}$$

Analytical gradient with respect to weight matrix $W$:
$$\frac{\partial \mathcal{L}}{\partial W} = \frac{1}{N} X^T (\hat{P} - Y)$$
Where $\hat{P}$ is the predicted probability matrix and $Y$ is the one-hot encoded ground truth.

---

### 2. Scratch Python Implementation: Softmax Regressor

In [ ]:
import numpy as np

class SoftmaxRegressionScratch:
    def __init__(self, lr=0.1, n_iter=1000):
        self.lr = lr
        self.n_iter = n_iter
        self.W = None
        self.b = None
        
    def _softmax(self, z):
        # Subtract max for numerical stability (prevents exp overflow)
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))
        
        # Initialize parameters
        np.random.seed(42)
        self.W = np.random.normal(0, 0.01, (n_features, n_classes))
        self.b = np.zeros(n_classes)
        
        # One-hot encode ground truth targets
        Y_onehot = np.zeros((n_samples, n_classes))
        Y_onehot[np.arange(n_samples), y] = 1.0
        
        for epoch in range(self.n_iter):
            # Forward pass: Logits -> Softmax probabilities
            logits = np.dot(X, self.W) + self.b
            P = self._softmax(logits)
            
            # Error residual: (P - Y)
            error = P - Y_onehot
            
            # Gradient computation
            dW = np.dot(X.T, error) / n_samples
            db = np.mean(error, axis=0)
            
            # Gradient descent step
            self.W -= self.lr * dW
            self.b -= self.lr * db
            
        return self
        
    def predict_proba(self, X):
        return self._softmax(np.dot(X, self.W) + self.b)
        
    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

# Generate 3-Class Synthetic Dataset
np.random.seed(42)
c1 = np.random.normal(loc=[-2, -2], scale=0.8, size=(100, 2))
c2 = np.random.normal(loc=[2, 2], scale=0.8, size=(100, 2))
c3 = np.random.normal(loc=[-2, 2], scale=0.8, size=(100, 2))

X_data = np.vstack([c1, c2, c3])
y_data = np.array([0]*100 + [1]*100 + [2]*100)

model = SoftmaxRegressionScratch(lr=0.5, n_iter=800).fit(X_data, y_data)
preds = model.predict(X_data)
accuracy = np.mean(preds == y_data) * 100

print("--- Softmax Multi-Class Regression ---")
print(f"Dataset Size:        {len(X_data)} samples across 3 classes")
print(f"Weight Matrix Shape: {model.W.shape}")
print(f"Training Accuracy:   {accuracy:.1f}%")

### 3. Step-by-Step Code Walkthrough
1. **Numerical Stability**: Subtracts row-wise maxima before calculating `np.exp` to avoid NaN overflow.
2. **Compact Matrix Gradient**: Calculates multi-class gradients as a single dot product $\frac{1}{N} X^T (\hat{P} - Y)$.
3. **Argmax Decision Rule**: Assigns the class with the highest posterior probability.

---

### 4. Industry Applications & Trade-offs
- **Output Layer of Deep Neural Networks:** Softmax regression forms the final classification head of LLMs and Vision Transformers.
- **Trade-off:** Assumes the **Independence of Irrelevant Alternatives (IIA)**; adding an identical duplicate class can artificially split probability mass.

---

### 5. Top Interview Questions & Answers
**Q1: What is the difference between One-vs-Rest (OvR) Binary Classifiers and Softmax Regression?**
> **Answer:** OvR trains $K$ independent binary classifiers whose probabilities do not sum to 1. Softmax regression jointly trains a single model where class probabilities are normalized across all classes via the softmax denominator.

**Q2: What is the Independence of Irrelevant Alternatives (IIA) property?**
> **Answer:** IIA states that the relative probability ratio between two choices is independent of whether a third alternative is present. If violated (the Red Bus / Blue Bus problem), multinomial logit produces biased estimates.